<h1>Chapter 5 | Data Exercise #3 | <code>hotels-europe</code> | Generalizing from Data</h1>
<h2>Introduction:</h2>
<p>In this notebook, you will find my notes and code for Chapter 5's <b>exercise 3</b> of the book <a href="https://gabors-data-analysis.com/">Data Analysis for Business, Economics, and Policy</a>, by Gábor Békés and Gábor Kézdi. The question was: 
<p>3. Use <code>hotels-europe</code> dataset and pick two cities and the same date.
<p>Assignments:</p>
<ul>
    <li>In each city, take hotels with three stars and calculate the average price.</li>
    <li>Estimate the standard error of the estimated average price by bootsrap and using the SE formula.</li>
    <li>Create 95% confidence intervals.</li>
    <li>Compare the average price and the confidence intervals across the two cities, and explain why you have a narrower CI for one city than the other..</li>


</ul>
<h2>1. Load the data</h2>

In [56]:
import os
import pandas as pd
import warnings
from datetime import datetime
from plotnine import *
import sys
import numpy as np
from scipy.stats import norm, sem
from numpy.random import choice
warnings.filterwarnings("ignore")

In [57]:
# Increase number of returned rows in pandas
pd.set_option("display.max_rows", 500)

In [58]:
# Current script folder
dirname = os.getcwd()

# Get location folders
data_in = f"{dirname}/da_data_repo/hotels-europe/clean/"
data_out = f"{dirname}/da_data_exercises/ch05-generalizing_from_data/03-hotels_europe/data/clean/"
output = f"{dirname}/da_data_exercises/ch05-generalizing_from_data/03-hotels_europe/data/output/"
func = f"{dirname}/da_case_studies/ch00-tech_prep/"
sys.path.append(func)
paths = [data_in, data_out, output]

for path in paths:
    if not os.path.exists(path):
        os.makedirs(path)

In [59]:
# Import the prewritten helper functions 
from py_helper_functions import *

# Get the data
**Command**: take hotels with three stars from two cities.

First, let's take a look at the data.

In [60]:
data_europe = pd.read_csv(f"{data_in}hotels-europe_price.csv")

In [61]:
data_europe.head()

,hotel_id,price,offer,offer_cat,year,month,weekend,holiday,nnights,scarce_room
0,1,172,0,0% no offer,2017,11,1,0,1,0
1,1,122,1,15-50% offer,2018,1,1,0,1,0
2,1,122,1,15-50% offer,2017,12,0,1,1,0
3,1,552,1,1-15% offer,2017,12,0,1,4,0
4,1,122,1,15-50% offer,2018,2,1,0,1,0


We probably need to do some data manipulation to get our features, which probably are at `hotels-europe_features.csv`.

In [62]:
hotels_europe_features = pd.read_csv(f"{data_in}hotels-europe_features.csv", header=0, skiprows=[1])

In [63]:
hotels_europe_features

,hotel_id,city,distance,stars,rating,country,city_actual,rating_reviewcount,center1label,center2label,neighbourhood,ratingta,ratingta_count,distance_alter,accommodation_type
0,3,Amsterdam,1.5,4.0,4.1,Netherlands,Amsterdam,165.0,City centre,Montelbaanstoren,Amsterdam,4.0,674.0,1.4,Hotel
1,4,Amsterdam,1.9,3.0,3.5,Netherlands,Amsterdam,298.0,City centre,Montelbaanstoren,Amsterdam,3.5,1882.0,2.1,Hotel
2,5,Amsterdam,1.8,3.5,4.0,Netherlands,Amsterdam,4.0,City centre,Montelbaanstoren,Amsterdam,4.5,66.0,2.0,Hotel
3,6,Amsterdam,1.9,4.0,4.1,Netherlands,Amsterdam,310.0,City centre,Montelbaanstoren,Amsterdam,4.0,767.0,2.0,Hotel
4,7,Amsterdam,0.8,3.5,4.4,Netherlands,Amsterdam,258.0,City centre,Montelbaanstoren,Amsterdam,4.5,273.0,1.2,Hotel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22896,19109,Rome,0.4,NaN,3.9,Italy,Rome,68.0,City centre,Grotta del Bue Marino,Trevi Fountain,4.0,100.0,0.7,Bed and breakfast
22897,19114,Rome,0.4,4.0,4.3,Italy,Rome,168.0,City centre,Palazzo Madama,Trevi Fountain,4.0,212.0,0.7,Hotel
22898,19115,Rome,0.4,NaN,4.3,Italy,Rome,44.0,City centre,Grotta del Bue Marino,Trevi Fountain,4.0,512.0,0.7,Guest House
22899,1,Amsterdam,3.1,4.0,4.3,Netherlands,Amsterdam,1030.0,City centre,Montelbaanstoren,Amsterdam,4.0,1115.0,3.6,Hotel


Ok, we'll work on this table first, select two cities and then join it with our facts table. We'll use Amsterdam and Madrid.

In [64]:
selected_cities = ["Amsterdam", "Madrid"]
hotels_cut_direct = hotels_europe_features[hotels_europe_features["city_actual"].isin(selected_cities)]

In [75]:
hotels_cut_direct["city_actual"].unique()

array(['Amsterdam', 'Madrid'], dtype=object)

In [66]:
hotels_cut_direct.head()

,hotel_id,city,distance,stars,rating,country,city_actual,rating_reviewcount,center1label,center2label,neighbourhood,ratingta,ratingta_count,distance_alter,accommodation_type
0,3,Amsterdam,1.5,4.0,4.1,Netherlands,Amsterdam,165.0,City centre,Montelbaanstoren,Amsterdam,4.0,674.0,1.4,Hotel
1,4,Amsterdam,1.9,3.0,3.5,Netherlands,Amsterdam,298.0,City centre,Montelbaanstoren,Amsterdam,3.5,1882.0,2.1,Hotel
2,5,Amsterdam,1.8,3.5,4.0,Netherlands,Amsterdam,4.0,City centre,Montelbaanstoren,Amsterdam,4.5,66.0,2.0,Hotel
3,6,Amsterdam,1.9,4.0,4.1,Netherlands,Amsterdam,310.0,City centre,Montelbaanstoren,Amsterdam,4.0,767.0,2.0,Hotel
4,7,Amsterdam,0.8,3.5,4.4,Netherlands,Amsterdam,258.0,City centre,Montelbaanstoren,Amsterdam,4.5,273.0,1.2,Hotel


In [76]:
# Recommended approach for your analysis
# Clean, readable, and performant
hotels_cut = hotels_europe_features[
    (hotels_europe_features["city_actual"].isin(selected_cities)) &
    (hotels_europe_features["accommodation_type"] == "Hotel") &
    (hotels_europe_features["stars"] == 3.0)
]

print("Final filtered dataset for analysis:")
print(f"Total hotels: {len(hotels_cut)}")
print("\nBreakdown by city:")
print(hotels_cut.groupby('city').size())

# Let's also check what columns we have available for the next steps
print(f"\nColumns available: {list(hotels_cut.columns)}")

Final filtered dataset for analysis:
Total hotels: 176

Breakdown by city:
city
Amsterdam    109
Madrid        67
dtype: int64

Columns available: ['hotel_id', 'city', 'distance', 'stars', 'rating', 'country', 'city_actual', 'rating_reviewcount', 'center1label', 'center2label', 'neighbourhood', 'ratingta', 'ratingta_count', 'distance_alter', 'accommodation_type']


We can now join both datasets.

In [77]:
hotels_df = pd.merge(hotels_cut, data_europe, on="hotel_id", how="left")

In [78]:
hotels_df

,hotel_id,city,distance,stars,rating,country,city_actual,rating_reviewcount,center1label,center2label,...,accommodation_type,price,offer,offer_cat,year,month,weekend,holiday,nnights,scarce_room
0,4,Amsterdam,1.9,3.0,3.5,Netherlands,Amsterdam,298.0,City centre,Montelbaanstoren,...,Hotel,115,1,15-50% offer,2017,12,0,1,1,0
1,4,Amsterdam,1.9,3.0,3.5,Netherlands,Amsterdam,298.0,City centre,Montelbaanstoren,...,Hotel,71,1,50%-75% offer,2017,11,0,0,1,0
2,11,Amsterdam,2.2,3.0,3.7,Netherlands,Amsterdam,341.0,City centre,Montelbaanstoren,...,Hotel,97,1,15-50% offer,2017,11,0,0,1,0
3,11,Amsterdam,2.2,3.0,3.7,Netherlands,Amsterdam,341.0,City centre,Montelbaanstoren,...,Hotel,167,0,0% no offer,2017,11,1,0,1,0
4,11,Amsterdam,2.2,3.0,3.7,Netherlands,Amsterdam,341.0,City centre,Montelbaanstoren,...,Hotel,105,1,15-50% offer,2018,1,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1114,9554,Madrid,3.2,3.0,3.9,Spain,Madrid,208.0,City centre,Casa Matesanz,...,Hotel,259,1,50%-75% offer,2017,12,0,1,4,0
1115,9554,Madrid,3.2,3.0,3.9,Spain,Madrid,208.0,City centre,Casa Matesanz,...,Hotel,95,1,15-50% offer,2018,5,1,0,1,0
1116,9554,Madrid,3.2,3.0,3.9,Spain,Madrid,208.0,City centre,Casa Matesanz,...,Hotel,90,1,15-50% offer,2018,4,1,0,1,0
1117,9563,Madrid,2.9,3.0,4.0,Spain,Madrid,42.0,City centre,Casa Matesanz,...,Hotel,78,1,1-15% offer,2018,5,1,0,1,1


We can now clean the data and return only our values of interest:
- Attributes
- Stars
- Price
- Same date

In [79]:
hotels_df["year"].max()

np.int64(2018)

In [91]:
hotels_cut = hotels_df[
    (hotels_df["city_actual"].isin(selected_cities)) &
    (hotels_df["accommodation_type"] == "Hotel") &
    (hotels_df["stars"] == 3.0) &
    (hotels_df["year"] == 2018)

]

In [95]:
hotels_cut = hotels_cut.loc[:, ["city_actual", "stars", "price"]]

In [96]:
hotels_cut

,city_actual,stars,price
4,Amsterdam,3.0,105
5,Amsterdam,3.0,117
9,Amsterdam,3.0,139
11,Amsterdam,3.0,139
14,Amsterdam,3.0,175
...,...,...,...
1113,Madrid,3.0,84
1115,Madrid,3.0,95
1116,Madrid,3.0,90
1117,Madrid,3.0,78


## Challenge #2 – Estimate the standard error of the estimated average price by bootsrap and using the SE formula.